# Stepper Motor Tutorial for STM32F429I-DISC1

Welcome to the complete stepper motor tutorial! This guide covers everything from basic concepts to advanced control techniques for stepper motors on your STM32F429I-DISC1 board.

## Table of Contents
1. Beginner Tutorial - Getting Started
2. Complete Tutorial - Advanced Control

---

## Part 1: Stepper Motor Beginner Tutorial

### What is a Stepper Motor?

A stepper motor is a **brushless DC motor** that divides a full rotation into a number of equal steps. Unlike regular motors that spin continuously, steppers move in precise, discrete steps.

### How Stepper Motors Work

**Basic Components:**
- **Stator**: Stationary electromagnets (coils)
- **Rotor**: Permanent magnet or toothed iron
- **Coils**: Electromagnets that create magnetic fields

**Step Sequence:**
Stepper motors work by energizing coils in a specific sequence to create rotation. For a bipolar stepper (4 wires), the sequence is:

```
Step 1: Coil A+, Coil B-  → 0°
Step 2: Coil A+, Coil B+  → 90°
Step 3: Coil A-, Coil B+  → 180°
Step 4: Coil A-, Coil B-  → 270°
```

### L298N Motor Driver

The L298N is an H-bridge driver that controls stepper motors using 4 input pins (IN1-IN4).

**L298N Pin Functions:**
- **IN1-IN4**: Logic inputs from STM32
- **OUT1-OUT2**: Connected to stepper coil A
- **OUT3-OUT4**: Connected to stepper coil B
- **VCC**: Motor power (5V-12V)
- **GND**: Ground

### Hardware Setup

**Connections for STM32F429I-DISC1:**

```
STM32 PC7  → L298N IN1  (Arduino D0)
STM32 PC6  → L298N IN2  (Arduino D1)
STM32 PG6  → L298N IN3  (Arduino D2)
STM32 PB4  → L298N IN4  (Arduino D3)

L298N OUT1 → Stepper Coil A+
L298N OUT2 → Stepper Coil A-
L298N OUT3 → Stepper Coil B+
L298N OUT4 → Stepper Coil B-

L298N VCC → 5V-12V Power Supply
L298N GND → GND
```

### Basic Code Example

```c
#include "stepper.h"

// Initialize stepper motor
STEPPER_Handle_t hstepper;
TIM_HandleTypeDef htim3;

STEPPER_Pins_t pins = {
    .port1 = GPIOC, .pin1 = GPIO_PIN_7,   // IN1 (PC7)
    .port2 = GPIOC, .pin2 = GPIO_PIN_6,   // IN2 (PC6)
    .port3 = GPIOG, .pin3 = GPIO_PIN_6,   // IN3 (PG6)
    .port4 = GPIOB, .pin4 = GPIO_PIN_4    // IN4 (PB4)
};

STEPPER_Init(&hstepper, &htim3, &pins);

// Move stepper motor
STEPPER_MoveSteps(&hstepper, 200, STEPPER_DIR_CW, 60);  // 1 revolution CW at 60 RPM
HAL_Delay(1000);
STEPPER_MoveSteps(&hstepper, 200, STEPPER_DIR_CCW, 60); // 1 revolution CCW
```

### Testing Your Setup

1. **Power Check**: Ensure L298N VCC is connected to motor power
2. **Connections**: Double-check all wire connections
3. **Code**: Upload and run the basic example
4. **Movement**: Motor should rotate smoothly in both directions

### Common Beginner Issues

- **Motor not moving**: Check power supply and coil connections
- **Erratic movement**: Verify step sequence and GPIO pins
- **Overheating**: L298N may need heatsink for continuous use

---

## Part 2: Stepper Motor Complete Tutorial

### Advanced Stepper Concepts

#### Step Modes

**Full Step Mode:**
- 4 steps per sequence
- Maximum torque
- Simple control

**Half Step Mode:**
- 8 steps per sequence
- Smoother movement
- Better resolution

**Microstepping:**
- Uses PWM to create intermediate steps
- Very smooth movement
- Requires advanced drivers (not L298N)

#### Speed and Acceleration

**Speed Control:**
- RPM (Revolutions Per Minute)
- Steps per second
- Delay between steps

**Acceleration Profiles:**
- Constant speed
- Linear acceleration/deceleration
- S-curve acceleration

### Advanced Hardware Setup

#### Current Limiting
The L298N has built-in current limiting. Adjust the potentiometer for your motor's current rating.

#### Enable Pins
Some L298N modules have EN pins. Connect to STM32 for software motor enable/disable.

#### Multiple Motors
You can control multiple steppers by using different GPIO pins and creating multiple handle instances.

### Advanced Code Examples

#### Speed Control

```c
// Set different speeds
STEPPER_SetSpeed(&hstepper, 30);   // Slow speed
STEPPER_MoveSteps(&hstepper, 100, STEPPER_DIR_CW, 30);

STEPPER_SetSpeed(&hstepper, 120);  // Fast speed
STEPPER_MoveSteps(&hstepper, 100, STEPPER_DIR_CW, 120);
```

#### Continuous Rotation

```c
// Rotate continuously
STEPPER_Rotate(&hstepper, STEPPER_DIR_CW);   // Start continuous CW rotation
HAL_Delay(5000);                             // Run for 5 seconds
STEPPER_Stop(&hstepper);                      // Stop rotation
```

#### Position Control

```c
// Move to absolute positions
STEPPER_MoveToPosition(&hstepper, 400, 60);  // Move to position 400 at 60 RPM
STEPPER_MoveToPosition(&hstepper, 0, 60);    // Return to home (position 0)
```

#### Configuration

```c
// Configure stepper parameters
STEPPER_Config_t config = STEPPER_GetDefaultConfig();
config.stepsPerRevolution = 400;    // 0.9° step angle
config.maxSpeedRPM = 200;           // Max 200 RPM
config.acceleration = 100;          // Acceleration steps/sec²

STEPPER_Config(&hstepper, &config);
```

### Step Sequences in Detail

#### Full Step Sequence (4 steps)

```c
// GPIO states for each step
const uint8_t fullStepSequence[4][4] = {
    {1, 1, 0, 0},  // Step 0: A+, B-
    {0, 1, 1, 0},  // Step 1: A-, B+
    {0, 0, 1, 1},  // Step 2: A-, B-
    {1, 0, 0, 1}   // Step 3: A+, B+
};
```

#### Half Step Sequence (8 steps)

```c
const uint8_t halfStepSequence[8][4] = {
    {1, 0, 0, 0},  // Step 0
    {1, 1, 0, 0},  // Step 1
    {0, 1, 0, 0},  // Step 2
    {0, 1, 1, 0},  // Step 3
    {0, 0, 1, 0},  // Step 4
    {0, 0, 1, 1},  // Step 5
    {0, 0, 0, 1},  // Step 6
    {1, 0, 0, 1}   // Step 7
};
```

### Timer Configuration

The stepper driver uses TIM3 for precise timing between steps:

```c
// Timer setup for microsecond delays
TIM_Base_Init(&htim3, 84, 0xFFFF);  // 1MHz timer clock
```

### Error Handling

```c
// Check for errors
STEPPER_StatusTypeDef status = STEPPER_MoveSteps(&hstepper, 200, STEPPER_DIR_CW, 60);
if (status != STEPPER_OK) {
    // Handle error
    switch (status) {
        case STEPPER_INVALID_PARAM:
            // Invalid parameters
            break;
        case STEPPER_BUSY:
            // Motor is busy
            break;
        default:
            // Other error
            break;
    }
}
```

### Performance Optimization

#### Interrupt-Driven Stepping
For smooth operation, use timer interrupts instead of delays:

```c
// In timer interrupt callback
void HAL_TIM_PeriodElapsedCallback(TIM_HandleTypeDef *htim) {
    if (htim->Instance == TIM3) {
        // Perform next step
        STEPPER_MoveStep(&hstepper);
    }
}
```

#### Buffer Management
For complex movements, implement a command buffer:

```c
typedef struct {
    uint32_t steps;
    STEPPER_Direction_t direction;
    uint16_t speed;
} STEPPER_Command_t;

// Command queue
STEPPER_Command_t commandBuffer[10];
uint8_t bufferHead = 0;
uint8_t bufferTail = 0;
```

### Troubleshooting Advanced Issues

#### Lost Steps
- **Cause**: Motor overload or too high speed
- **Solution**: Reduce speed, check power supply, add mechanical load compensation

#### Resonance
- **Cause**: Motor vibrates at certain speeds
- **Solution**: Use microstepping or avoid resonant frequencies

#### Torque Issues
- **Cause**: Insufficient current or wrong coil connections
- **Solution**: Check L298N current limit, verify wiring

### Applications

#### CNC Machines
Precise positioning for X/Y/Z axes

#### 3D Printers
Stepper motors control print head and build plate movement

#### Robotics
Joint control in robotic arms

#### Camera Sliders
Smooth camera movement for video

### Best Practices

1. **Power Supply**: Use appropriate voltage/current for your motor
2. **Heat Management**: Monitor L298N temperature
3. **Mechanical**: Ensure proper mounting and belt/gear ratios
4. **Software**: Implement proper error handling and limits
5. **Testing**: Start slow and gradually increase speed

### Further Reading

- Stepper motor theory and mathematics
- PID control for position accuracy
- Advanced driver ICs (DRV8825, TMC2208)
- Closed-loop stepper systems

---

## Conclusion

You've now learned both basic and advanced stepper motor control! From simple rotation to complex positioning systems, you have the knowledge to implement stepper motors in your STM32F429I-DISC1 projects.

**Key Takeaways:**
- Stepper motors provide precise positioning
- L298N driver handles the power requirements
- Speed and acceleration control enable smooth movement
- Proper error handling ensures reliable operation

Experiment with different motors, speeds, and configurations to find what works best for your application! 🔧🤖